# Ders 4: Öz-Denetimli Öğrenme ve Temsil Öğrenme

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 3 ve 6 (Kayıp fonksiyonları, ESA'lar).

Etiket pahalıdır, ham veri değildir. Öz-denetimli öğrenme, etiketsiz bir temsil öğrenebilmek için
görevi verinin kendisinden icat eder. Bu defterde önemli üç aileyi ele alıyoruz — **karşıtsal**
(InfoNCE, SimCLR, CLIP), **karşıtsal olmayan** (BYOL ve kaçınmak zorunda olduğu çökme problemi) ve
**maskeli yeniden kurulum** (MAE) — ve davranışlarının büyük kısmını açıklayan iki özelliği:
*hizalanma* (alignment) ve *tekdüzelik* (uniformity).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100

def l2norm(x, axis=-1):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + 1e-12)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

print("Kütüphaneler yüklendi.")


## 1. InfoNCE: Etiketsiz Bir Sınıflandırma Problemi

Bir çapa $x$, bir **pozitif** $x^+$ (aynı öğenin başka bir görünümü) ve $N-1$ **negatif** alın.
InfoNCE modelden, pozitifi yığın içinden seçmesini ister:

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(z, z^+)/\tau)}
{\exp(\text{sim}(z, z^+)/\tau) + \sum_{j} \exp(\text{sim}(z, z_j^-)/\tau)} .$$

Bu tam olarak, "etiketi" yığın içindeki bir konum olan $N$ sınıflı bir çapraz entropi problemidir;
dolayısıyla hiçbir etiketlemeye gerek yoktur. Bundan iki sonuç doğrudan çıkar:

- Kayıp, aşağıdan $\log N$ eksi karşılıklı bilgi ile sınırlıdır — **daha çok negatif daha sıkı bir
  sınır** demektir; karşıtsal öğrenmede yığın boyutunun bu kadar önemli olmasının nedeni budur.
- $\tau$ sıcaklığı negatiflerin ne kadar sert ağırlıklandırılacağını belirler. Küçük $\tau$, gradyanı
  en yakın (en zor) negatiflerde yoğunlaştırır.


In [ ]:
def info_nce(z, z_pos, z_neg, tau=0.1):
    s_pos = z @ z_pos / tau
    s_neg = z_neg @ z / tau
    return -(s_pos - np.log(np.exp(s_pos) + np.exp(s_neg).sum()))

d = 32
z      = l2norm(np.random.randn(d))
z_pos  = l2norm(z + 0.35*np.random.randn(d))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

Ns = np.array([1, 2, 4, 8, 16, 64, 256, 1024, 4096])
losses = [np.mean([info_nce(z, z_pos, l2norm(np.random.randn(N, d)), 0.1) for _ in range(20)]) for N in Ns]
axes[0].semilogx(Ns, losses, "o-", lw=2, label="InfoNCE kaybı")
axes[0].semilogx(Ns, np.log(Ns+1), "--", lw=2, c="crimson", label="karşılıklı bilgi için log(N+1) üst sınırı")
axes[0].set_xlabel("negatif sayısı N"); axes[0].set_ylabel("kayıp")
axes[0].set_title("Daha çok negatif -> daha sıkı sınır"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

sims = np.linspace(-1, 1, 200)
for tau, c in zip([0.5, 0.2, 0.07, 0.02], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    w = softmax(sims/tau)
    axes[1].plot(sims, w/w.max(), lw=2, c=c, label=f"tau={tau}")
axes[1].set_xlabel("bir negatifin kosinüs benzerliği"); axes[1].set_ylabel("göreli gradyan ağırlığı")
axes[1].set_title("Sıcaklık hangi negatiflerin önemli olduğuna karar verir")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Tau'nun öğrenilen geometriye etkisi, sabit bir yığın üzerinde ölçülüyor
B = 128
anchors = l2norm(np.random.randn(B, d))
for tau in [0.02, 0.1, 0.5]:
    S = anchors @ anchors.T / tau
    np.fill_diagonal(S, -np.inf)
    axes[2].hist(softmax(S, axis=1).max(1), bins=30, alpha=0.6, label=f"tau={tau}")
axes[2].set_xlabel("en zor tek negatife verilen ağırlık"); axes[2].set_ylabel("sayı")
axes[2].set_title("Küçük tau = zor negatif madenciliği"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()


## 2. Hizalanma ve Tekdüzelik

Karşıtsal öğrenme aynı anda iki iş yapar ve bunlar ayrı ayrı ölçülebilir:

$$\mathcal{L}_{\text{hiza}} = \mathbb{E}\lVert z - z^+ \rVert^2, \qquad
\mathcal{L}_{\text{tekdüze}} = \log \mathbb{E}\, e^{-2\lVert z_i - z_j \rVert^2}.$$

**Hizalanma** aynı öğenin iki görünümünü birbirine çeker; **tekdüzelik** ise tüm öğeleri hiperküre
üzerine yayarak temsilin mümkün olduğunca çok bilgi tutmasını sağlar. Yalnızca pozitif terim her şeyi
tek bir noktaya çökertirdi; iticiliği negatifler sağlar. Aşağıda çember üzerinde, hizalanmaları aynı
ama tekdüzelikleri çok farklı üç temsil var.


In [ ]:
def alignment(z1, z2):
    return np.mean(np.sum((z1-z2)**2, axis=1))

def uniformity(z, t=2.0):
    d2 = ((z[:, None, :] - z[None, :, :])**2).sum(-1)
    iu = np.triu_indices(len(z), 1)
    return np.log(np.mean(np.exp(-t*d2[iu])))

n = 200
th_unif  = np.random.uniform(0, 2*np.pi, n)
th_clump = np.random.normal(0, 0.35, n)
th_two   = np.where(np.random.rand(n) < 0.5, np.random.normal(0, .2, n), np.random.normal(np.pi, .2, n))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, th) in zip(axes, [("tekdüze (iyi)", th_unif),
                                 ("çökmüş (kötü)", th_clump),
                                 ("iki küme", th_two)]):
    z = np.stack([np.cos(th), np.sin(th)], 1)
    z_pos = l2norm(z + 0.05*np.random.randn(n, 2))
    ax.scatter(z[:, 0], z[:, 1], s=12, alpha=0.6)
    circ = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(circ), np.sin(circ), c="k", lw=0.8, alpha=0.4)
    ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_title(f"{name}\nhiza={alignment(z, z_pos):.3f}   tekdüze={uniformity(z):.3f}", fontsize=11)
plt.suptitle("Aynı hizalanma, çok farklı tekdüzelik", fontsize=13)
plt.tight_layout(); plt.show()
print("Daha düşük (daha negatif) tekdüzelik = gömme kürenin daha çoğunu kullanıyor = daha çok bilgi korunuyor.")


## 3. SimCLR: Veri Artırımı Değişmezlikleri Belirler

SimCLR $B$ görüntülük bir yığın alır, her birinin iki artırılmış görünümünü üretir ve bu $2B$
görünümü, her görünümün kardeşini bulmak zorunda olduğu $2B$ sınıflı bir sınıflandırma problemi
olarak ele alır. Ortaya çıkan temsille ilgili her şeyi artırım politikası belirler: **neyi
artırırsanız model ona karşı değişmez hâle gelir**. Renk oynaması modeli renge karşı değişmez yapar —
nesne tanıma için mükemmel, kırmızı elmayı yeşilinden ayırt etmesi gereken bir görev için ölümcül.


In [ ]:
def simclr_loss(Z1, Z2, tau=0.5):
    Z = l2norm(np.vstack([Z1, Z2]))
    B = len(Z1)
    S = Z @ Z.T / tau
    np.fill_diagonal(S, -1e9)
    targets = np.concatenate([np.arange(B, 2*B), np.arange(0, B)])   # kardeş görünümün indeksi
    P = softmax(S, axis=1)
    return -np.log(P[np.arange(2*B), targets] + 1e-12).mean(), S

B, d = 8, 16
base = l2norm(np.random.randn(B, d))
Z1 = l2norm(base + 0.15*np.random.randn(B, d))
Z2 = l2norm(base + 0.15*np.random.randn(B, d))
loss, S = simclr_loss(Z1, Z2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
im = axes[0].imshow(np.where(S < -1e8, np.nan, S), cmap="viridis")
axes[0].set_title(f"2B görünümün benzerlik matrisi (kayıp={loss:.3f})")
axes[0].set_xlabel("görünüm j"); axes[0].set_ylabel("görünüm i"); plt.colorbar(im, ax=axes[0])
for i in range(2*B):
    j = (i+B) % (2*B)
    axes[0].scatter([j], [i], s=25, facecolors="none", edgecolors="crimson", lw=1.5)

# Artırım şiddeti: çok zayıf = önemsiz görev, çok güçlü = içeriği yok eder
strengths = np.linspace(0.0, 1.4, 25)
losses, task_info = [], []
for s in strengths:
    A = l2norm(base + s*np.random.randn(B, d))
    Bv = l2norm(base + s*np.random.randn(B, d))
    losses.append(simclr_loss(A, Bv)[0])
    task_info.append(np.mean(np.sum(l2norm(base)*A, axis=1)))    # içeriğin ne kadarı hayatta kalıyor
axes[1].plot(strengths, losses, lw=2, label="SimCLR kaybı (görev zorluğu)")
axes[1].plot(strengths, task_info, lw=2, label="korunan içerik (orijinale kosinüs)")
axes[1].set_xlabel("artırım şiddeti"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_title("Artırımın ideal aralığı")

# Değişmezlik artırımdan miras alınır
labels = ["kırpma", "renk oynatma", "bulanıklaştırma", "döndürme", "gri tonlama"]
useful = [[0.9, 0.8, 0.5, 0.3, 0.7],      # nesne tanıma
          [0.8, 0.1, 0.4, 0.2, 0.1]]      # ince taneli renk görevi
x = np.arange(len(labels))
axes[2].bar(x-0.2, useful[0], 0.4, label="nesne tanıma")
axes[2].bar(x+0.2, useful[1], 0.4, label="renge bağlı görev")
axes[2].set_xticks(x); axes[2].set_xticklabels(labels, rotation=20, fontsize=9)
axes[2].set_ylabel("bu değişmezliğin yararlılığı")
axes[2].set_title("Görevden bağımsız bir artırım politikası yoktur"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()


## 4. Çökme ve BYOL'un Negatifsiz Çözümü

InfoNCE'den negatifleri çıkarın; $\lVert z - z^+\rVert^2$ kaybının kusursuz ama işe yaramaz bir çözümü
vardır: her girdiyi aynı sabite eşlemek. **BYOL** ve **SimSiam** bunu hiç negatif kullanmadan, iki
asimetriyle önler:

1. Yalnızca çevrimiçi (online) dalda bir **yordayıcı** (predictor) başlık.
2. Ağırlıkları çevrimiçi ağırlıkların üstel hareketli ortalaması olan hedef dalda bir
   **gradyan durdurma** (stop-gradient).

Böylece hedef, çevrimiçi ağın kovaladığı ama içinden türev almadığı yavaş hareket eden bir nesne olur
ve çökmüş çözüm kararlı bir sabit nokta olmaktan çıkar. Aşağıdaki benzetim bunu doğrusal bir
kodlayıcının kendi EMA kopyasını kovalamasına indirger ve temsilin **etkin rankını** (kovaryans
spektrumunun katılım oranı) ölçer: BYOL düzeneğiyle rank korunur, düzenek olmadan 1'e doğru düşer.


In [ ]:
# Paylaşılan DOĞRUSAL bir kodlayıcı f(v) = normalize(W v). Her girdinin iki gürültülü görünümü.
def norm_jac(g, z, yn):                     # L2 normalleştirmesi üzerinden geri yayılım
    return (g - (g*z).sum(1, keepdims=True)*z) / yn[:, None]

def eff_rank(Z):                            # kovaryans spektrumunun katılım oranı
    s = np.linalg.svd(Z - Z.mean(0), compute_uv=False)**2
    return float((s.sum()**2)/(s**2).sum())

def train(mode, d=16, n=256, steps=2000, lr=1.0, ema=0.99, seed=0):
    rng = np.random.default_rng(seed)
    X  = rng.normal(size=(n, d))
    W  = np.eye(d) + 0.05*rng.normal(size=(d, d))     # çevrimiçi kodlayıcı
    Wt = W.copy()                                     # EMA hedef kodlayıcı
    P  = np.eye(d) + 0.05*rng.normal(size=(d, d))     # yordayıcı başlık
    hist = []
    for _ in range(steps):
        v1 = X + 0.3*rng.normal(size=X.shape)
        v2 = X + 0.3*rng.normal(size=X.shape)
        y1 = v1 @ W.T; z1 = l2norm(y1); n1 = np.linalg.norm(y1, axis=1)

        if mode == "yalnızca hizalanma":                      # simetrik, yordayıcı yok, gradyan durdurma yok
            y2 = v2 @ W.T; z2 = l2norm(y2); n2 = np.linalg.norm(y2, axis=1)
            g  = 2*(z1 - z2)
            W -= lr*(norm_jac(g, z1, n1).T @ v1 + norm_jac(-g, z2, n2).T @ v2)/n
        else:                                         # BYOL: yordayıcı + gradyan durdurma + EMA hedefi
            yp = z1 @ P.T; p = l2norm(yp); pn = np.linalg.norm(yp, axis=1)
            z2 = l2norm(v2 @ Wt.T)                    # <- bu dala gradyan akmaz
            gp = norm_jac(2*(p - z2), p, pn)
            P -= lr*(gp.T @ z1)/n
            W -= lr*(norm_jac(gp @ P, z1, n1).T @ v1)/n
            Wt = ema*Wt + (1-ema)*W
        hist.append(eff_rank(l2norm(X @ W.T)))
    return np.array(hist)

h_align = train("yalnızca hizalanma")
h_byol  = train("byol")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(h_align, lw=2, label="yalnızca hizalanma kaybı")
axes[0].plot(h_byol,  lw=2, label="yordayıcı + gradyan durdurma + EMA (BYOL)")
axes[0].axhline(1.0, ls="--", c="crimson", lw=1.2, label="rank 1 = tam çökme")
axes[0].set_xlabel("adım"); axes[0].set_ylabel("temsilin etkin rankı")
axes[0].set_title("Çökme yalnızca kaybın değil, dinamiğin bir özelliğidir")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

methods = ["SimCLR", "MoCo", "BYOL", "SimSiam", "Barlow Twins", "VICReg"]
needs   = ["yığın içi negatifler", "negatifler + momentum kuyruğu", "yordayıcı + EMA hedefi",
           "yordayıcı + gradyan durdurma", "çapraz korelasyon -> birim", "varyans + kovaryans terimleri"]
axes[1].axis("off"); axes[1].set_title("Her yöntem çökmeyi nasıl engelliyor", fontsize=12)
for i, (m, nd) in enumerate(zip(methods, needs)):
    axes[1].text(0.02, 0.88-0.15*i, f"{m:14s}<-  {nd}", fontsize=10.5, family="monospace",
                 transform=axes[1].transAxes)

plt.tight_layout(); plt.show()
print(f"eğitim sonrası etkin rank:  yalnızca-hizalanma {h_align[-1]:.2f}   |   BYOL {h_byol[-1]:.2f}")
print("Yalnızca hizalanma ile eğitilen kodlayıcı, neredeyse sabit hâle gelene dek yönleri atar.")


## 5. Maskeli Otokodlayıcı (MAE)

Diğer aile karşılaştırmak yerine yeniden kurar. MAE görüntü yamalarının büyük bir kısmını maskeler
(tipik olarak %75) ve bir kod çözücüden pikselleri yeniden inşa etmesini ister. İki tasarım tercihi
bunu çalışır kılar:

- **Maske oranı yüksek olmalıdır.** %15'te (metin için BERT varsayılanı) bir yama komşularından
  kolayca aradeğerlenir ve model anlam yerine yerel doku öğrenir. Görüntüler metinden çok daha
  fazla artıklık (redundancy) taşır.
- **Kodlayıcı yalnızca görünür yamaları görür.** %75 maskeli durumda bu 4 kat daha ucuz bir
  kodlayıcı demektir; ön eğitimi karşılanabilir kılan da budur.


In [ ]:
# Oyuncak "görüntüler": düşük frekanslı 2B desenler, yamalara ayrılmış
def make_image(k=16, seed=None):
    rng = np.random.default_rng(seed)
    xs = np.linspace(0, 3, k)
    a, b, c = rng.normal(size=3)
    return np.sin(a*xs[:, None] + b*xs[None, :]) + 0.5*np.cos(c*xs[:, None]*xs[None, :])

def patchify(img, p=4):
    k = img.shape[0]
    return img.reshape(k//p, p, k//p, p).transpose(0, 2, 1, 3).reshape(-1, p*p)

def unpatchify(P, k=16, p=4):
    g = k//p
    return P.reshape(g, g, p, p).transpose(0, 2, 1, 3).reshape(k, k)

# "Kod çözücü": görünür yamalardan maskeli yamalara ridge regresyon, eğitim kümesinde uydurulmuş
p, k = 4, 16
train = np.stack([patchify(make_image(k, s), p) for s in range(600)])       # (N, T, p*p)
N, T, D = train.shape

def mae_experiment(mask_ratio, seed=0):
    rng = np.random.default_rng(seed)
    n_mask = max(1, int(round(mask_ratio*T)))
    idx = rng.permutation(T)
    mask_idx, vis_idx = idx[:n_mask], idx[n_mask:]
    Xtr = train[:500][:, vis_idx].reshape(500, -1)
    Ytr = train[:500][:, mask_idx].reshape(500, -1)
    Xte = train[500:][:, vis_idx].reshape(100, -1)
    Yte = train[500:][:, mask_idx].reshape(100, -1)
    A = np.linalg.solve(Xtr.T@Xtr + 1e-2*np.eye(Xtr.shape[1]), Xtr.T@Ytr)
    err = np.mean((Xte@A - Yte)**2) / np.mean(Yte**2)
    return err, mask_idx, vis_idx, A, Xte, Yte

ratios = [0.15, 0.25, 0.50, 0.75, 0.875]
errs = [mae_experiment(r)[0] for r in ratios]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].plot(ratios, errs, "o-", lw=2)
axes[0].set_xlabel("maske oranı"); axes[0].set_ylabel("göreli yeniden kurulum hatası")
axes[0].set_title("Düşük maske oranı = fazlasıyla kolay görev"); axes[0].grid(alpha=0.3)

err, mask_idx, vis_idx, A, Xte, Yte = mae_experiment(0.75)
img = train[500].copy()
masked = img.copy(); masked[mask_idx] = 0.0
pred = img.copy(); pred[mask_idx] = (Xte[0:1] @ A).reshape(-1, D)
for ax, (title, arr) in zip(axes[1:], [("orijinal", img), ("%75 maskeli girdi", masked),
                                       ("yeniden kurulum", pred)]):
    ax.imshow(unpatchify(arr, k, p), cmap="RdBu_r", vmin=-1.8, vmax=1.8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"kodlayıcı {T} yamanın {len(vis_idx)} tanesini görüyor -> ~{T/len(vis_idx):.1f} kat ucuz ileri geçiş")
print("Metin düşük maske oranı (%15), görüntü yüksek maske oranı (%75) ister: artıklık farklıdır.")


## 6. CLIP: İki Kipiliğe Yayılan Karşıtsal Öğrenme

CLIP, "aynı görüntünün iki görünümü" yerine "bir görüntü ve onun altyazısı" ikilisini kullanır. İki
kodlayıcı ortak bir uzaya eşler ve $B \times B$ benzerlik matrisi üzerine **simetrik** bir InfoNCE
kaybı uygulanır — köşegen pozitiflerdir.

Kazanç sıfır-atış (zero-shot) sınıflandırmadır: sınıf adlarını metin olarak kodlayın ("bir {sınıf}
fotoğrafı"), görüntüyü kodlayın ve en yakın metin gömmesini seçin. Hiçbir sınıflandırma başlığı
eğitilmediğinden yeni bir sınıfın maliyeti yeni bir cümleden ibarettir.


In [ ]:
d, B = 24, 12
concepts = l2norm(np.random.randn(B, d))                       # paylaşılan anlamsal içerik
W_img = l2norm(concepts + 0.25*np.random.randn(B, d))          # görüntü kodlayıcı çıktısı
W_txt = l2norm(concepts + 0.25*np.random.randn(B, d))          # metin kodlayıcı çıktısı

tau = 0.07
S = W_img @ W_txt.T / tau
loss_i = -np.log(softmax(S, 1)[np.arange(B), np.arange(B)]).mean()
loss_t = -np.log(softmax(S, 0)[np.arange(B), np.arange(B)]).mean()
print(f"CLIP kaybı = (görüntü->metin {loss_i:.3f} + metin->görüntü {loss_t:.3f}) / 2 = {(loss_i+loss_t)/2:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
im = axes[0].imshow(W_img @ W_txt.T, cmap="viridis")
axes[0].set_xlabel("metin gömmesi"); axes[0].set_ylabel("görüntü gömmesi")
axes[0].set_title("Benzerlik matrisi: köşegen pozitif çiftlerdir")
plt.colorbar(im, ax=axes[0])

# Görülmemiş "sınıflar" ile sıfır-atış sınıflandırma
classes = ["kedi", "köpek", "araba", "ağaç", "tekne"]
txt_emb = l2norm(np.random.randn(len(classes), d))
true = 2
img_emb = l2norm(txt_emb[true] + 0.45*np.random.randn(d))
probs = softmax(img_emb @ txt_emb.T / 0.07)
axes[1].bar(classes, probs, color=["crimson" if i == true else "steelblue" for i in range(len(classes))])
axes[1].set_ylabel("sıfır-atış olasılığı")
axes[1].set_title(f"Sıfır-atış tahmini: {classes[int(np.argmax(probs))]}  (gerçek: {classes[true]})")

# Kipilik boşluğu: görüntü ve metin gömmeleri ayrı koniler işgal eder
proj = np.random.randn(d, 2)
pi, pt = W_img @ proj, W_txt @ proj
axes[2].scatter(pi[:, 0], pi[:, 1], s=45, label="görseller")
axes[2].scatter(pt[:, 0], pt[:, 1], s=45, marker="s", label="metinler")
for a, b in zip(pi, pt):
    axes[2].plot([a[0], b[0]], [a[1], b[1]], c="gray", lw=0.7, alpha=0.6)
axes[2].set_title("Eşleşmiş gömmeler (rastgele 2B izdüşüm)"); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("Gerçek CLIP modellerinde iki kipilik kürenin ayrı konilerinde durur -- 'kipilik boşluğu'.")
print("Sıfır-atış sınıflandırma için önemli olan, bir kipilik içindeki GÖRELİ sıralamadır.")


## 7. Bir Temsili Değerlendirmek

Öz-denetimli bir kodlayıcı tahmin üretmez; bu yüzden üzerine *basit* bir başlığın ne kadar kolay
oturduğuna bakılarak değerlendirilir:

| Protokol | Ölçtüğü şey |
|---|---|
| **Doğrusal sonda** | Bilgi doğrusal olarak erişilebilir mi? Standart manşet sayısı. |
| **k-NN sondası** | Metrik yapının kendisi anlamlı mı? Hiç eğitim yok. |
| **İnce ayar** | Üst sınır; ama kısmen temsili değil ilk değeri ölçer. |
| **Az-atışlı (few-shot)** | Örnek verimliliği — pratikte genellikle asıl önemli sayı. |

Yararlı bir teşhis, öznitelik kovaryansının **özdeğer spektrumudur**: varyansı birkaç yöne çökmüş
bir temsil (boyutsal çökme), eğitim kaybı sağlıklı görünse bile doğrusal sondada kötü sonuç verir.


In [ ]:
d, n = 64, 2000
good     = np.random.randn(n, d)                                  # tam ranklı öznitelikler
collapsed = np.random.randn(n, d) @ np.diag(np.exp(-np.arange(d)/4.0))   # rank birkaç yöne yoğunlaşmış

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for name, Z in [("sağlıklı", good), ("boyutsal olarak çökmüş", collapsed)]:
    s = np.linalg.svd(Z - Z.mean(0), compute_uv=False)**2
    s = s/s.sum()
    axes[0].semilogy(s, lw=2, label=f"{name} (etkin rank {(s.sum()**2)/(s**2).sum():.1f})")
axes[0].set_xlabel("bileşen"); axes[0].set_ylabel("açıklanan varyans (log)")
axes[0].set_title("Öznitelik kovaryans spektrumu"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# İki temsil için, etiketli veri miktarına göre doğrusal sonda doğruluğu
def probe_acc(Z, n_lab):
    w_true = np.random.randn(Z.shape[1])
    y = (Z @ w_true > 0).astype(float)
    Xtr, ytr, Xte, yte = Z[:n_lab], y[:n_lab], Z[1500:], y[1500:]
    w = np.linalg.solve(Xtr.T@Xtr + 1e-3*np.eye(Z.shape[1]), Xtr.T@(2*ytr-1))
    return np.mean((Xte@w > 0) == (yte > 0))

labels = [10, 25, 50, 100, 250, 500, 1000]
for name, Z in [("sağlıklı", good), ("çökmüş", collapsed)]:
    accs = [np.mean([probe_acc(Z, m) for _ in range(15)]) for m in labels]
    axes[1].semilogx(labels, accs, "o-", lw=2, label=name)
axes[1].set_xlabel("etiketli örnek sayısı"); axes[1].set_ylabel("doğrusal sonda doğruluğu")
axes[1].set_title("Az-atışlı davranış ikisini ayırıyor"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 8. Özet

| Kavram | Açıklama |
|---|---|
| **InfoNCE** | Yığın üzerinde çapraz entropi; "etiket" hangi öğenin kardeş görünüm olduğudur |
| **$\log N$ sınırı** | Karşılıklı bilgi sınırı negatif sayısıyla iyileşir → büyük yığınlar |
| **Sıcaklık $\tau$** | Küçük $\tau$ gradyanı en zor negatiflerde yoğunlaştırır |
| **Hizalanma / tekdüzelik** | Pozitifler birbirine çeker, negatifler küreye yayar |
| **SimCLR** | İki artırılmış görünüm, $2B$ sınıflı örnek ayırt etme |
| **Artırım politikası** | Değişmezlikleri belirler; görevden bağımsız bir seçim yoktur |
| **Çökme** | Negatifsiz bir kaybın aşikâr minimumu |
| **BYOL / SimSiam** | Yordayıcı + gradyan durdurma + EMA hedefi çökmeyi kararsız kılar |
| **MAE** | Yüksek maske oranı (%75); kodlayıcı yalnızca görünür yamaları görür |
| **CLIP** | Görüntü–metin çiftleri üzerinde simetrik InfoNCE; sıfır-atış aktarımı sağlar |
| **Değerlendirme** | Doğrusal / k-NN / az-atışlı sondalar; öznitelik spektrumunu izleyin |

**Sonraki Defter →** Aktarımlı Öğrenme ve Parametre-Verimli İnce Ayar
